# TRADUCCIÓN DE LENGUAJE DE SEÑAS MEXICANAS

***Este Jupyter tiene los contenidos acerca del proyecto de lenguaje de señas mexicanas.***


Una problematica muy común que se ha pensado desde siempre es que todas las personas puedan ser capaces de entenderse mutuamente sin complicaciones, es decir, ya sea que manejemos el mismo idioma, modismos o derivados, siempre se ha querido que nos entendemos por completo.
Antes la mayor problemática era con los lenguajes, ya que existen demasiados lenguajes, en cada país generalmente se habla un lenguaje distinto, sino, puede hablarse uno distinto en cada estado, hay casos donde hasta en cada ciudad que necesites ir es un dialecto completamente distinto, como el caso de Arabia Saudita. Sin embargo, con el uso de las redes profundas, IA y ML, ya no hay tanto ese problema, ya que está resolviendose en estos tiempos y teniendo resultados muy buenos. Esto ocasiona que podamos irnos a otras cuestiones, como es nuestro caso, irnos directamente a un problema sin solución.


¿Cómo podemos hacer para poder resolver el que una persona sorda o sordo-muda y una persona con capacidades del habla, sean capaces de hablar si uno no conoce el dialecto del otro?

Este es el problema que intentaremos resolver, específicamente a la hora de realizar traducción de señas estáticas, es decir, señas en donde no se realice ningún movimiento para poder reconocerlas, como en el caso de 'A', 'B', 'O', etc. En contrario como con las señas dinámicas como la 'J', la 'Q' o la 'Z'.

Ahora mismo **(En proceso)** contiene los contenidos de la tesis escrita por Ricardo Fernando **Morfin** Chávez.
"Reconocimiento continuo de la lengua de señas mexicanas".

# ***---Configuraciones---***

In [2]:
DEBUG = False
CREATE_KEYPOINTS = False
CREATE_MODEL = False
EVALUATE_MODEL = False

# ***--- Librerías ---***


In [ ]:
from __future__ import annotations

import json
import os
import random
import re
from dataclasses import dataclass
import pathlib
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import csv

import mediapipe as mp        
import joblib                 # guardar/cargar modelos u objetos
import matplotlib.pyplot as plt
import seaborn as sns       

# ---------- Scikit-learn ----------
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
)


from sklearn.model_selection import (
    GridSearchCV,
    GroupKFold,
    ParameterGrid,
    StratifiedKFold,
    train_test_split,
    cross_validate,
)

try:
    from sklearn.model_selection import StratifiedGroupKFold  
except Exception:
    StratifiedGroupKFold = None

from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# ***---LSM---***

### --- Uso de carpetas --- 

In [ ]:
DATA_DIR = Path(r"MSL-ABC\lsm-abc-A\test\A")

image_paths = list(DATA_DIR.rglob("*.jpg"))

data = []
for p in image_paths:
    try:
        with Image.open(p) as img:
            w, h = img.size
            mode = img.mode
    except:
        w, h, mode = None, None, "corrupt"
    label = p.parent.name 
    data.append([str(p), label, w, h, mode])

df = pd.DataFrame(data, columns=["path", "label", "width", "height", "mode"])


### --- Visualizar distribución de imagenes ---


In [ ]:
plt.figure(figsize=(10,5))
sns.countplot(data=df, x='label', order=df['label'].value_counts().index)
plt.title("Número de imágenes por clase")
plt.xticks(rotation=90)
plt.show()

### --- Distribución de tamaño ---


In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(df['width'], df['height'], alpha=0.3)
plt.xlabel("Width")
plt.ylabel("Height")
plt.title("Distribución de tamaños de imágenes")
plt.show()

### --- Ejemplos de imágenes ---

In [ ]:
def show_examples(label, n=9):
    samples = df[df['label']==label].sample(min(n, len(df[df['label']==label])))
    plt.figure(figsize=(8,8))
    for i, (_, row) in enumerate(samples.iterrows()):
        plt.subplot(3,3,i+1)
        img = Image.open(row['path'])
        plt.imshow(img)
        plt.axis("off")
    plt.suptitle(f"Ejemplos de clase: {label}")
    plt.show()

In [ ]:
# Ejemplo con la clase A
show_examples("A")

# ***--- Uso de MediaPipe para uso en tiempo real ---***


### --- Uso de camara para creación de keypoints ---

In [ ]:
def real_time_keypoints():

    mp_hands = mp.solutions.hands
    mp_draw  = mp.solutions.drawing_utils
    mp_style = mp.solutions.drawing_styles

    root = pathlib.Path(mp.__file__).parent
    asset = root / "modules" / "hand_landmark" / "hand_landmark_tracking_cpu.binarypb"
    if not asset.exists():
        print("Falta el asset de MediaPipe:", asset)
        return

    import cv2
    for idx in [0,1,2,3]:
        cap = cv2.VideoCapture(idx)
        if cap.isOpened():
            print(f"Usando cámara índice {idx}")
            break
        cap.release()
    else:
        print("No se pudo abrir ninguna cámara.")
        return

    with mp_hands.Hands(
        model_complexity=1,
        max_num_hands=2,
        min_detection_confidence=0.5, #¿Más menos? ¿Qué pasa?
        min_tracking_confidence=0.5 #¿Más menos? ¿Qué pasa?
    ) as hands:
        while True:
            ok, frame = cap.read()
            frame = cv2.flip(frame, 1)
            if not ok:
                break
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = hands.process(rgb)
            if res.multi_hand_landmarks:
                for hand_lms, handedness in zip(res.multi_hand_landmarks, res.multi_handedness):
                    mp_draw.draw_landmarks(
                        frame, hand_lms, mp_hands.HAND_CONNECTIONS,
                        mp_style.get_default_hand_landmarks_style(),
                        mp_style.get_default_hand_connections_style()
                    )
                    label = handedness.classification[0].label
                    score = handedness.classification[0].score
                    
                    h, w = frame.shape[:2]
                    x0 = int(hand_lms.landmark[mp_hands.HandLandmark.WRIST].x * w)
                    y0 = int(hand_lms.landmark[mp_hands.HandLandmark.WRIST].y * h)
                    cv2.putText(frame, f"{label} {score:.2f}", (x0, y0-10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
                    
            cv2.imshow("Hands (q para salir)", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    cap.release()
    cv2.destroyAllWindows()


### --- Aplicación de uso real ---

In [ ]:
if __name__ == "__main__" and DEBUG == True:
    real_time_keypoints()

# ***---Entrenamiento de modelo ---***

### Configuraciones para modelado de keypoints

In [ ]:
DATA_ROOT = Path(r"MSL-ABC\lsm-abc-B")
KEYPOINTS_ROOT = Path(r"MSL-ABC\keypoints")
MODEL_ROOT = Path(r"MSL-ABC\svm_models")

TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR  = DATA_ROOT / "test"

OUT_TRAIN_CSV = KEYPOINTS_ROOT /"kp_train_B.csv"
OUT_TEST_CSV  = KEYPOINTS_ROOT / "kp_test_B.csv"
MODEL_PATH    = MODEL_ROOT / "svm_kp_B.joblib"

mph = mp.solutions.hands
mp_draw  = mp.solutions.drawing_utils
mp_style = mp.solutions.drawing_styles

### Función de normalización de puntos

In [ ]:
def normalize_pts_minmax(pts: np.ndarray):

    X = np.asarray(pts, dtype=np.float32)  # (21,2)
    x = X[:, 0]
    y = X[:, 1]
    xr = x.max() - x.min()
    yr = y.max() - y.min()
    xr = xr if xr > 1e-6 else 1e-6  # evitar división por 0
    yr = yr if yr > 1e-6 else 1e-6
    x_n = (x - x.min()) / xr
    y_n = (y - y.min()) / yr
    P = np.stack([x_n, y_n], axis=1)  # (21,2)
    return P.reshape(-1)              # (42,)

### Extraccion de keypoints en la imagen

In [ ]:
def extract_keypoints_from_image(img_path, hands) -> np.ndarray | None:
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    h, w = img.shape[:2]
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    res = hands.process(rgb)
    if not res.multi_hand_landmarks:
        return None
    lm = res.multi_hand_landmarks[0].landmark
    pts = np.array([[p.x * w, p.y * h] for p in lm], dtype=np.float32)
    return normalize_pts_minmax(pts)  # (42,)

### Ejemplo arbitrario con Keypoints

In [ ]:
if DEBUG == True:
    example_path = r'MSL-ABC\lsm-abc-C\train\B\S1-B-5-0.jpg'

    img = cv2.imread(example_path)
    if img is None:
        raise FileNotFoundError(f"No se pudo leer: {example_path}")

    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    with mph.Hands(static_image_mode=True, max_num_hands=1, model_complexity=1) as hands:
        res = hands.process(rgb)

    if res.multi_hand_landmarks:
        for hand_lms in res.multi_hand_landmarks:
            mp_draw.draw_landmarks(
                rgb, hand_lms, mph.HAND_CONNECTIONS,
                mp_style.get_default_hand_landmarks_style(),
                mp_style.get_default_hand_connections_style()
            )

    plt.figure(figsize=(6,6))
    plt.imshow(rgb)
    plt.axis("off")
    plt.title("Imagen con keypoints")
    plt.show()

### Creación de CSV con keypoints

In [ ]:
def build_csv_from_split(split_dir: Path, out_csv: Path):
    """Recorre directorios y crea un CSV con columnas: label, f0..f41, path."""
    classes = sorted([d.name for d in split_dir.iterdir() if d.is_dir()])
    rows = []
    with mph.Hands(static_image_mode=True, max_num_hands=1, model_complexity=1) as hands:
        for cls in classes:
            cls_dir = split_dir / cls
            imgs = list(cls_dir.rglob("*.jpg")) + list(cls_dir.rglob("*.png")) + list(cls_dir.rglob("*.jpeg"))
            for p in tqdm(imgs, desc=f"{split_dir.name}/{cls}", unit="img"):
                vec = extract_keypoints_from_image(p, hands)
                if vec is None:
                    continue
                rows.append([cls, *vec.tolist(), str(p)])
    # Guardar
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    cols = ["label"] + [f"f{i}" for i in range(42)] + ["path"]
    pd.DataFrame(rows, columns=cols).to_csv(out_csv, index=False)
    print(f"✔ CSV guardado: {out_csv}  ({len(rows)} filas)")

### Entrenamiento de SVM con RBF

In [ ]:
def train_svm(train_csv: Path, model_path: Path):
    df = pd.read_csv(train_csv)
    X = df[[f"f{i}" for i in range(42)]].values.astype(np.float32)
    y = df["label"].values
    clf = make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", C=10, gamma="scale", probability=True, class_weight="balanced")
    )
    clf.fit(X, y)
    joblib.dump(clf, model_path)
    print(f"✔ Modelo SVM guardado en: {model_path}")
    return clf

In [ ]:
def evaluate_svm(model, test_csv: Path):
    df = pd.read_csv(test_csv)
    X = df[[f"f{i}" for i in range(42)]].values.astype(np.float32)
    y = df["label"].values
    y_pred = model.predict(X)
    print("=== Reporte ===")
    print(classification_report(y, y_pred, digits=3))
    print("=== Matriz de confusión ===")
    print(confusion_matrix(y, y_pred))

### Proceso completo de Creación de keypoints y evaluación de SVM

In [ ]:
if __name__ == "__main__":

    if CREATE_KEYPOINTS == True:
        build_csv_from_split(TRAIN_DIR, OUT_TRAIN_CSV)
        build_csv_from_split(TEST_DIR,  OUT_TEST_CSV)
    
    if CREATE_MODEL == True:
        model = train_svm(OUT_TRAIN_CSV, MODEL_PATH)

    else:
        model = None

    if EVALUATE_MODEL == True:
        evaluate_svm(model, OUT_TEST_CSV)

### Evaluación manual SVM

In [ ]:
# Se debe de cambiar las rutas dependiendo de si qué grupo o variación es

if DEBUG == True:
    OUT_TRAIN_CSV = r"MSL-ABC\keypoints\kp_train_C.csv"
    OUT_TEST_CSV  = r"MSL-ABC\keypoints\kp_test_C.csv"

    if model == None:
        model = r'MSL-ABC\svm_models\svm_kp_C.joblib' 
    evaluate_svm(model, OUT_TEST_CSV)

# GRID SEARCH SVM

### Configuración

In [ ]:
DATA_ROOT = Path(r"MSL-ABC\keypoints")
RESULT_ROOT = Path(r"MSL-ABC\gridsearch")
MODEL_ROOT = Path(r"MSL-ABC\gridsearch\models_svm")

TRAIN_CSV = DATA_ROOT / "kp_train_C.csv"   
TEST_CSV  = DATA_ROOT / "kp_test_C.csv"        

OUT_CV_RESULTS = MODEL_ROOT / "svm_cv_results_C.csv"
OUT_BEST_SUMMARY = MODEL_ROOT / "svm_best_summary_C.csv"

MODELS_DIR = MODEL_ROOT

mph = mp.solutions.hands
mp_draw  = mp.solutions.drawing_utils
mp_style = mp.solutions.drawing_styles

### Carga de puntos

In [ ]:
def load_xy(csv_path: Path):
    df = pd.read_csv(csv_path)
    feat_cols = [f"f{i}" for i in range(42)]  # 42 = 21 landmarks * (x,y)
    X = df[feat_cols].to_numpy(dtype=np.float32)
    y = df["label"].to_numpy()
    return X, y  

#### Verificación de existencia de Test_CSV

In [ ]:
X_train, y_train = load_xy(TRAIN_CSV)

has_test = TEST_CSV.exists()
if has_test:
    X_test, y_test = load_xy(TEST_CSV)
    print("CSV EXISTENTE")
else:
    print("CSV NO EXISTE, SE USARÁ PARTE DEL TRAIN COMO SPLITTING PARA TESTING")
    # Si no hay test, separa un 20% del train para evaluación final
    X_train, X_test, y_train, y_test = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
    )

## Grid Search

In [ ]:
pipe = make_pipeline(
    StandardScaler(),
    SVC(class_weight="balanced", probability=True)
)

### Parametros a buscar

In [ ]:
"""param_grid = [
    # LINEAR
    {
        "svc__kernel": ["linear"],
        "svc__C": [0.1, 1, 3, 10, 30, 100],
    },
    # RBF
    {
        "svc__kernel": ["rbf"],
        "svc__C": [0.1, 1, 3, 10, 30, 100],
        "svc__gamma": ["scale", 0.01, 0.03, 0.1, 0.3, 1.0],
    },
    # POLY
    {
        "svc__kernel": ["poly"],
        "svc__degree": [2, 3],
        "svc__C": [0.1, 1, 3, 10],
        "svc__gamma": ["scale", 0.03, 0.1],
        "svc__coef0": [0.0, 0.5, 1.0],
    },
    # SIGMOID
    {
        "svc__kernel": ["sigmoid"],
        "svc__C": [0.1, 1, 3, 10, 30],
        "svc__gamma": ["scale", 0.01, 0.1],
        "svc__coef0": [0.0, 0.5, 1.0],
    },
]"""

param_grid = [

        # POLY
    {
        "svc__kernel": ["poly"],
        "svc__degree": [2, 3],
        "svc__C": [0.1, 1, 3, 10],
        "svc__gamma": ["scale", 0.03, 0.1],
        "svc__coef0": [0.0, 0.5, 1.0],
    }
]

In [ ]:
grid = list(ParameterGrid(param_grid))
print(f"Total de combinaciones: {len(grid)}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "f1_macro": "f1_macro",
    "accuracy": "accuracy",
}

manual_results = []

### Búsqueda de Cross-Validation

In [ ]:
for params in tqdm(grid, desc="GridSearch", unit="comb"):
    pipe.set_params(**params)
    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1,           
        error_score="raise"
    )
    mean_f1 = float(scores["test_f1_macro"].mean())
    std_f1  = float(scores["test_f1_macro"].std())
    mean_acc = float(scores["test_accuracy"].mean())
    std_acc  = float(scores["test_accuracy"].std())
    manual_results.append({**params,
                           "mean_f1_macro": mean_f1, "std_f1_macro": std_f1,
                           "mean_accuracy": mean_acc, "std_accuracy": std_acc})

manual_df = (pd.DataFrame(manual_results)
             .sort_values("mean_f1_macro", ascending=False)
             .reset_index(drop=True))

manual_out = MODEL_ROOT / "svm_cv_results_poly.csv"
manual_df.to_csv(manual_out, index=False)
print(f"✔ Resultados guardados en: {manual_out}")

In [ ]:
best_row = manual_df.iloc[0]
best_params_manual = {k: best_row[k] for k in best_row.index if k.startswith("svc__")}
print("[MANUAL] Mejor set:", best_params_manual)
print("[MANUAL] CV mean_f1_macro:", best_row["mean_f1_macro"])

best_pipe_manual = clone(pipe).set_params(**best_params_manual)
best_pipe_manual.fit(X_train, y_train)

In [ ]:
if "X_test" in globals() and "y_test" in globals() and X_test is not None:
    y_pred_m = best_pipe_manual.predict(X_test)
    print("\n=== [MANUAL] TEST Classification Report ===")
    print(classification_report(y_test, y_pred_m, digits=4))
    cm_m = confusion_matrix(y_test, y_pred_m)
    cmn_m = confusion_matrix(y_test, y_pred_m, normalize="true")
    pd.DataFrame(cm_m).to_csv(MODEL_ROOT / "svm_confusion_matrix_raw_MANUAL_poly.csv", index=False)
    pd.DataFrame(cmn_m).to_csv(MODEL_ROOT / "svm_confusion_matrix_norm_MANUAL_poly.csv", index=False)

model_path_manual = MODEL_ROOT / "svm_best_model_MANUAL_poly.joblib"

## Lectura de 

In [ ]:
df_cm = pd.read_csv(r'MSL-ABC\gridsearch\models_svm\svm_confusion_matrix_raw_MANUAL_poly.csv', header = None)

# Quitar solo la primera fila
conf_mat = df_cm.iloc[1:].astype(int)

In [ ]:
labels = [chr(65 + i) for i in range(conf_mat.shape[0])]

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(conf_mat, annot=True, fmt="d", cmap="YlGnBu",
            xticklabels=labels, yticklabels=labels, cbar=True)

plt.title("Matriz de confusión - SVM", fontsize=18)
plt.xlabel("Predicciones", fontsize=14)
plt.ylabel("Clases verdaderas", fontsize=14)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
pivot_table = df.pivot_table(
    index="svc__C", 
    columns="svc__gamma", 
)

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(pivot_table, annot=True, fmt=".4f", cmap="viridis")
plt.title("Heatmap F1-macro SVM (C vs gamma)")
plt.ylabel("C")
plt.xlabel("gamma")
plt.show()


In [ ]:
sns.heatmap(
    pivot_table, 
    annot=True, fmt=".4f", cmap="viridis",
    cbar_kws={'label': 'F1-macro'}
)
plt.title("Heatmap F1-macro SVM (C vs gamma)")
plt.ylabel("C (log)")
plt.xlabel("gamma (log)")
plt.show()

# ***---SEÑAS ESTATICAS + DINAMICAS---***

A la hora de usarlo en tiempo real, una aplicación no revisará fotos, sino que tendrá un sujeto que esté realizando pruebas o incluso ya teniendo una conversación, por lo que tenemos que involucrar **videos**.

A la hora de realizar estos videos, se utilizarán exactamente el **mismo dataset** como base para crear una **n cantidad de videos** con **m subgrupos**, haciendo una combinación de imágenes asemejando un video (Estilo caricatura cuando emulan el movimiento con varios frames que se van alternando, dando el efecto del movimiento), **no todos los frames pueden ser utilizados**, entonces se tomarán técnicas para poder ver **cuáles y cuántos frames** utilizar por video. 

En el paper dio como resultado que **30**, **10** y **2** frames serán las cantidades idoneas.

Los videos reconstruidos a base d las imagenes, se realizarán de 30 frames cada uno, derivando de qué frames se usarán, sin embargo a la hora de querer realizar pruebas con menos, se realizará un submuestreo del mismo video, y se emplearán técnicas para decidir qué frames se utilizarán 

# ***---Stacking + Padding ---***

### Configuración

In [1]:
from dataclasses import dataclass
from pathlib import Path
import re
from datetime import datetime


ROOTS = [
    Path("DB_LSM/MSL-dynamic-signs/train"),
    Path("DB_LSM/MSL-dynamic-signs/test"),
    # Agrega más si ocupas
]

EXTS = ("mp4", "mov", "avi")
DRY_RUN = True  # <-- Cambia a False cuando ya verifiques

### Objeto de clase video para manejo organizado

In [2]:
@dataclass
class VideoMeta:
    person_id: int
    label: str
    view: str
    take_id: int
    src_path: Path

### Expresión regular del formato de carpetas

In [3]:
RX_MAIN = re.compile(
    r"^S(?P<person>\d+)-(?P<label>[A-Za-z0-9]+)-(?P<view>[A-Za-z0-9_-]+)-(?P<take>\d+)\.(?:mp4|mov|avi)$",
    re.IGNORECASE,
)

TOKEN_FAIL = object()

### Función de búsqueda con uso de expresión regular

In [4]:
def parse_regex(p: Path):
    m = RX_MAIN.match(p.name)
    if not m:
        return None
    return VideoMeta(
        person_id=int(m["person"]),
        label=m["label"].upper(),
        view=m["view"].lower(),
        take_id=int(m["take"]),
        src_path=p,
    )

### Función de búsqueda con uso de tokens (Fallo por REGEX)

In [5]:
def parse_tokens(p: Path):
    stem = p.stem
    parts = stem.split("-")
    if len(parts) != 4:
        return TOKEN_FAIL
    p_raw, label, view, take = parts
    if not p_raw.lower().startswith("s"):
        return TOKEN_FAIL
    try:
        person_id = int(p_raw[1:])
        take_id = int(take)
    except ValueError:
        return TOKEN_FAIL
    return VideoMeta(
        person_id=person_id,
        label=label.upper(),
        view=view.lower(),
        take_id=take_id,
        src_path=p,
    )

### Unión

In [6]:
def parse_filename(p: Path):
    meta = parse_regex(p)
    if not meta:  # si regex no jaló, intenta tokens
        meta = parse_tokens(p)
        if meta is TOKEN_FAIL:
            return None
    return meta

#### Exploración de videos

In [ ]:
def discover(root: Path):
    valid = []
    invalid = []
    for vid in root.rglob("*.mp4"):
        meta = parse_filename(vid)
        if meta:
            valid.append(meta)
        else:
            invalid.append(vid)
    return valid, invalid


#### Aplicación de búsqueda de videos

In [11]:
if __name__ == "__main__":
    ROOT = Path(r"DB_LSM\MSL-dynamic-signs\train")
    if not ROOT.exists():
        print(f"❌ No existe: {ROOT.resolve()}")

    vids, bads = discover(ROOT)

    print(f"\n❌ Videos no válidos: {len(bads)}")
    print(f"✅ Videos válidos: {len(vids)}")

    for b in bads[:5]:
        print("No válido:", b.name)
    for v in vids[:5]:
        print("Válido:", v)



❌ Videos no válidos: 45
✅ Videos válidos: 517
No válido: S19-K-Frontal.1.mp4
No válido: S19-K-Frontal.2.mp4
No válido: S19-K-Frontal.3.mp4
No válido: S19-K-Frontal.4.mp4
No válido: S19-K-Frontal.5.mp4
Válido: VideoMeta(person_id=1, label='J', view='frontal', take_id=1, src_path=WindowsPath('DB_LSM/MSL-dynamic-signs/train/S1-J-frontal-1.mp4'))
Válido: VideoMeta(person_id=1, label='J', view='frontal', take_id=2, src_path=WindowsPath('DB_LSM/MSL-dynamic-signs/train/S1-J-frontal-2.mp4'))
Válido: VideoMeta(person_id=1, label='J', view='frontal', take_id=3, src_path=WindowsPath('DB_LSM/MSL-dynamic-signs/train/S1-J-frontal-3.mp4'))
Válido: VideoMeta(person_id=1, label='J', view='frontal', take_id=4, src_path=WindowsPath('DB_LSM/MSL-dynamic-signs/train/S1-J-frontal-4.mp4'))
Válido: VideoMeta(person_id=1, label='J', view='frontal', take_id=5, src_path=WindowsPath('DB_LSM/MSL-dynamic-signs/train/S1-J-frontal-5.mp4'))


En la primera iteración, antes de realizar cualquier cambio, habrá aproximadamente 45 erroneos, eso quiere decir que podemos aprovecharnos, cambiar el nombre y asegurarnos que sí se puedan utilizar, y cambiaremos el nombre para no perder esos datos.

In [12]:
RX_BAD = re.compile(
    r"^(S\d+)-([^.]+)\.([0-9]+)\.(mp4|mov|avi)$",
    re.IGNORECASE
)

### Función de cambio de nombre

In [13]:
def propose_new_name(name: str):

    if RX_MAIN.match(name):
        return None

    m = RX_BAD.match(name)
    if not m:
        return None

    sN, label_view, take, ext = m.groups()
    # solo cambiamos el separador '.' -> '-'
    new_name = f"{sN}-{label_view}-{take}.{ext.lower()}"
    return new_name

### Función completa de cambios de nombres

In [ ]:
def normalize_dirs(roots):
    changes = []
    skipped_exists = 0
    scanned = 0
    for root in roots:
        if not root.exists():
            print(f"⚠️  No existe: {root.resolve()}")
            continue

        for fp in root.rglob("*"):
            if not fp.is_file():
                continue
            if fp.suffix.lower().lstrip(".") not in EXTS:
                continue

            scanned += 1
            new_name = propose_new_name(fp.name)
            if not new_name:
                continue  # ya está OK o no aplica patrón

            dst = fp.with_name(new_name)
            if dst.exists():
                skipped_exists += 1
                print(f"⛔ Destino ya existe, se omite: {dst.name}")
                continue

            changes.append((fp, dst))

    print(f"\nArchivos escaneados: {scanned}")
    print(f"Renombramientos propuestos: {len(changes)}")
    print(f"Saltados por colisión: {skipped_exists}")

    ts = datetime.now().strftime("%Y%m%d-%H%M%S")
    log_path = Path(f"rename_log_{ts}.csv")
    with log_path.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["src", "dst"])
        for src, dst in changes:
            w.writerow([str(src), str(dst)])

    print(f"📄 Log guardado en: {log_path.resolve()}")

    # Apply
    if not changes:
        return

    if DRY_RUN:
        print("🔎 DRY_RUN=True → No se realizaron cambios. Si todo se ve bien, pon DRY_RUN=False.")
    else:
        for src, dst in changes:
            src.rename(dst)
        print(f"✅ Renombramientos aplicados: {len(changes)}")

In [ ]:
if __name__ == "__main__":
    normalize_dirs(ROOTS)

### Proceso Principal (Stacking)

### Aplicación

# Pooling

## Configuración

In [4]:
# === Raíces de datos ===
DATA_ROOT = Path("DB_LSM")
INPUT_DIRS = [
    DATA_ROOT / "MSL-dynamic-signs" / "train",
    # DATA_ROOT / "MSL-dynamic-profile-signs" / "test",
]

DERIV_ROOT = Path("derivatives")
CSV_DIR = DERIV_ROOT / "csv"        
SAVE_WINDOWS_AS_FRAMES = False      # auditoría visual opcional

# (Si SAVE_WINDOWS_AS_FRAMES=True, creamos FRAMES_OUT en main)
# FRAMES_OUT = DERIV_ROOT / "frames"

# === Ventanas y stride (50%) ===
WINDOWS = {30: 15, 10: 5, 5: 2, 2: 1}

FPS_TARGET = 30

QUALITY_THRESH = {
    "frames_ok_ratio_min": 0.8,   # % de frames con landmarks válidos
    "visible_hands_ratio_min": 0.0,  # pon 0.9 si quieres forzar manos visibles
}

# === Hands / landmarks ===
N_HAND_LANDMARKS = 21
FEATURE_CHANNELS = 2 * N_HAND_LANDMARKS 

# === Features estadísticos ===
FEATURE_STAT_NAMES = ["mean", "std", "min", "max", "p25", "p50", "p75", "dmean", "dstd"]

# === Metadatos  ===
META_COLS_FIXED = [
    "segment_uid",
    "label", "label_id",
    "person_id", "view", "video_id", "take_id",
    "window_size", "segment_idx", "start_frame", "end_frame",
    "stride", "fps", "fps_target",
    "width", "height",
    "frames_ok_ratio", "visible_hands_ratio", "left_present_ratio", "right_present_ratio",
    "low_quality",
    "src_video_path", "path_segment",
    "dataset_version", "extractor_version", "norm_type", "random_seed",
]

# === Salidas ===
CSV_META_NAME = "segments_metadata_train.csv"
CSV_FEAT_NAME = "segments_features_train.csv"


In [5]:
VIDEO_RE = re.compile(
    r"^S(?P<person>\d+)-(?P<label>[A-ZÑ])-?(?P<view>[a-zA-Z]+)-(?P<take>\d+)\.mp4$",
    re.UNICODE,
)
# Ej.: S20-J-frontal-1.mp4  -> person=20, label=J, view=frontal, take=1

In [6]:
@dataclass
class VideoMeta:
    person_id: int
    label: str
    view: str
    take_id: int
    video_id: str
    src_path: Path

In [7]:
def make_segment_uid(video_id: str, win: int, seg_idx: int) -> str:
    return f"{video_id}|w{win}|seg{seg_idx:04d}"

### Elección de columnas

In [8]:
def select_feature_columns(df_feat, mode: str = "all"):
    base = []
    if mode in ("all", "mean"): base += [c for c in df_feat.columns if c.startswith("mean_f")]
    if mode in ("all", "std"):  base += [c for c in df_feat.columns if c.startswith("std_f")]
    if mode == "all":
        base += [c for c in df_feat.columns if c.startswith(("min_f","max_f","p25_f","p50_f","p75_f","dmean_f","dstd_f"))]
    return ["segment_uid"] + base

### Construcción de metadatos

In [9]:
def build_meta_row(
    meta, *, segment_uid: str, win: int, seg_idx: int, a: int, b: int,
    width: int, height: int, fps_target: int,
    frames_ok_ratio: float, visible_hands_ratio: float,
    left_present_ratio: float, right_present_ratio: float,
    low_quality: int, base_seg_dir: Path, stride: int
) -> dict:
    """Arma la fila de METADATOS (sin features numéricas)."""
    return {
        "segment_uid": segment_uid,
        "label": meta.label,
        "person_id": meta.person_id,
        "view": meta.view,
        "video_id": meta.video_id,
        "take_id": meta.take_id,
        "window_size": win,
        "segment_idx": seg_idx,
        "start_frame": a,
        "end_frame": b - 1,
        "fps": float(fps_target),
        "width": width,
        "height": height,
        "frames_ok_ratio": frames_ok_ratio,
        "visible_hands_ratio": visible_hands_ratio,
        "left_present_ratio": left_present_ratio,
        "right_present_ratio": right_present_ratio,
        "low_quality": low_quality,
        "src_video_path": str(meta.src_path),
        "path_segment": str((base_seg_dir / f"seg_{seg_idx:03d}").as_posix()),
        "dataset_version": "v1",
        "extractor_version": "mp_hands_mc1",
        "norm_type": "bbox",
        "stride": stride,
        "fps_target": fps_target,
        "random_seed": 42,
    }


### Buscar el video por nombre REGEX

In [10]:
def parse_video_name(p: Path) -> Optional[VideoMeta]:
    m = VIDEO_RE.match(p.name)
    if not m:
        return None
    d = m.groupdict()
    person = int(d["person"])
    label = d["label"].upper()
    view = d["view"].lower()
    take = int(d["take"])
    vid = f"S{person}-{label}-{view}-{take}"
    return VideoMeta(person, label, view, take, vid, p)

### Construcción de features

In [11]:
def build_feat_row(segment_uid: str, feats: np.ndarray, feature_channels: int) -> dict:
    """Arma la fila de FEATURES puras (numéricas) + segment_uid para join."""
    row = {"segment_uid": segment_uid}
    for s_i, s in enumerate(FEATURE_STAT_NAMES):
        base = s_i * feature_channels
        for f_i in range(feature_channels):
            row[f"{s}_f{f_i}"] = float(feats[base + f_i])
    return row

### Lectura de metadatos

In [ ]:
def read_video_meta(cv_cap) -> Tuple[int, int, float, int]:
    w = int(cv_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cv_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = float(cv_cap.get(cv2.CAP_PROP_FPS) or 0.0)
    n = int(cv_cap.get(cv2.CAP_PROP_FRAME_COUNT))
    return w, h, fps, n

### Creación de índices para submuestreo

In [13]:
def sample_indices_to_target_fps(n_src: int, fps_src: float, fps_tgt: int) -> np.ndarray:
    """Crea índices de frames fuente para remuestrear al FPS objetivo."""
    if fps_src <= 1e-6 or n_src == 0:
        return np.arange(0)
    duration = n_src / fps_src
    n_desired = max(int(round(duration * fps_tgt)), 1)
    src_times = np.linspace(0, duration, num=n_src, endpoint=False)
    tgt_times = np.linspace(0, duration, num=n_desired, endpoint=False)
    idx = np.searchsorted(src_times, tgt_times, side="left")
    idx = np.clip(idx, 0, n_src - 1)
    return idx

### Ventaneo

In [14]:
def sliding_windows(n_frames: int, win: int, stride: int) -> List[Tuple[int, int]]:
    spans = []
    i = 0
    while i + win <= n_frames:
        spans.append((i, i + win))  # [start, end)
        i += stride
    return spans

## Extracción de Landmarks

In [15]:
def extract_landmarks_xy(mp_hands, img_bgr) -> Tuple[Optional[np.ndarray], Optional[str]]:
    """
    Devuelve:
      - landmarks 2D normalizados a bbox: shape (21, 2) o None si no hay mano
      - handedness: 'left' | 'right' | None (de la mano elegida)
    Estrategia: elegir la mano con bbox más compacto (aprox. mejor centrada/visible).
    """
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    res = mp_hands.process(img_rgb)
    if not res.multi_hand_landmarks:
        return None, None

    hands_lm = res.multi_hand_landmarks
    hands_hd = getattr(res, "multi_handedness", None)

    h, w, _ = img_bgr.shape
    best = None
    best_score = -1
    best_handed = None

    for idx, hlm in enumerate(hands_lm):
        xs = np.array([lm.x for lm in hlm.landmark])
        ys = np.array([lm.y for lm in hlm.landmark])
        x_min, x_max = xs.min(), xs.max()
        y_min, y_max = ys.min(), ys.max()
        x0 = int(np.clip(x_min * w, 0, w - 1))
        y0 = int(np.clip(y_min * h, 0, h - 1))
        x1 = int(np.clip(x_max * w, 0, w - 1))
        y1 = int(np.clip(y_max * h, 0, h - 1))
        bw = max(1, x1 - x0)
        bh = max(1, y1 - y0)
        # Score = inverso del área bbox (más compacto -> mejor)
        score = 1.0 / float(bw * bh)
        if score > best_score:
            best_score = score
            best = (hlm, (x0, y0, bw, bh))
            if hands_hd and idx < len(hands_hd):
                handed_label = hands_hd[idx].classification[0].label.lower()
                best_handed = 'left' if 'left' in handed_label else ('right' if 'right' in handed_label else None)
            else:
                best_handed = None

    if best is None:
        return None, None

    hlm, (x0, y0, bw, bh) = best
    xs = np.array([lm.x for lm in hlm.landmark]) * w
    ys = np.array([lm.y for lm in hlm.landmark]) * h
    x_norm = (xs - x0) / float(bw)
    y_norm = (ys - y0) / float(bh)
    coords = np.stack([x_norm, y_norm], axis=1)  # (21,2)
    return coords.astype(np.float32), best_handed


In [16]:
def pool_window_features(seq_xy: List[np.ndarray]) -> np.ndarray:
    """Pooling estadístico por canal para una ventana -> vector (9*42 = 378)."""
    T = len(seq_xy)
    if T == 0:
        return np.full((FEATURE_CHANNELS * 9,), np.nan, dtype=np.float32)

    arr = np.stack(seq_xy, axis=0).reshape(T, -1)  # (T, 42)
    delta = np.diff(arr, axis=0) if T >= 2 else np.zeros_like(arr)

    parts = [
        np.nanmean(arr, axis=0),
        np.nanstd(arr, axis=0),
        np.nanmin(arr, axis=0),
        np.nanmax(arr, axis=0),
        np.nanpercentile(arr, 25, axis=0),
        np.nanpercentile(arr, 50, axis=0),
        np.nanpercentile(arr, 75, axis=0),
        np.nanmean(np.abs(delta), axis=0),
        np.nanstd(delta, axis=0),
    ]
    return np.concatenate(parts, axis=0).astype(np.float32)

## Procesamiento de video (Proceso Principal)

In [21]:
def process_video(
    meta: VideoMeta,
    hands,
    *,
    windows: Dict[int, int] = None,          # {win: stride}
    fps_target: int = FPS_TARGET,
    quality_thresh: Dict[str, float] = None, # {"frames_ok_ratio_min": 0.8, "visible_hands_ratio_min": 0.0}
    feature_channels: int = FEATURE_CHANNELS,
    save_frames: bool = SAVE_WINDOWS_AS_FRAMES,
    frames_out_root: Optional[Path] = None,  # pasa FRAMES_OUT si save_frames=True
) -> Tuple[List[dict], List[dict]]:
    """Procesa un video -> listas de filas (metadata, features) por segmento."""
    if windows is None:
        windows = WINDOWS
    if quality_thresh is None:
        quality_thresh = QUALITY_THRESH

    meta_rows: List[dict] = []
    feat_rows: List[dict] = []

    cap = cv2.VideoCapture(str(meta.src_path))
    if not cap.isOpened():
        return meta_rows, feat_rows

    width, height, fps_src, n_src = read_video_meta(cap)
    idx_src = sample_indices_to_target_fps(n_src, fps_src, fps_target)
    if len(idx_src) == 0:
        cap.release()
        return meta_rows, feat_rows

    # --- lee frames remuestreados a fps_target ---
    current_src = -1
    frames_selected = []
    for target_i in idx_src:
        while current_src < target_i:
            ret = cap.grab()
            if not ret:
                break
            current_src += 1
        ret, frame = cap.retrieve()
        if not ret:
            break
        frames_selected.append(frame)
    cap.release()

    # --- landmarks por frame ---
    lm_per_frame: List[Optional[np.ndarray]] = []
    left_flag: List[int] = []
    right_flag: List[int] = []
    for f in frames_selected:
        lm, handed = extract_landmarks_xy(hands, f)
        lm_per_frame.append(lm)
        left_flag.append(1 if handed == "left" else 0)
        right_flag.append(1 if handed == "right" else 0)

    # --- ventanas ---
    for win, stride in windows.items():
        spans = sliding_windows(len(frames_selected), win, stride)

        # ruta base sólo si guardamos frames
        if save_frames:
            if frames_out_root is None:
                raise ValueError("save_frames=True pero frames_out_root es None")
            base_seg_dir = frames_out_root / f"win{win}" / meta.label / meta.video_id
        else:
            base_seg_dir = None  # no se usa

        for seg_idx, (a, b) in enumerate(spans):
            seg_lms = [lm for lm in lm_per_frame[a:b] if lm is not None]
            feats = pool_window_features(seg_lms)

            frames_ok_ratio = (len(seg_lms) / float(win)) if win > 0 else 0.0
            low_quality = int(frames_ok_ratio < quality_thresh["frames_ok_ratio_min"])
            left_present_ratio = float(np.mean(left_flag[a:b])) if (b - a) > 0 else 0.0
            right_present_ratio = float(np.mean(right_flag[a:b])) if (b - a) > 0 else 0.0
            visible_hands_ratio = float(
                np.mean([1 if lm_per_frame[i] is not None else 0 for i in range(a, b)])
            ) if (b - a) > 0 else 0.0

            # guardar frames para auditoría (opcional)
            if save_frames:
                seg_dir = base_seg_dir / f"seg_{seg_idx:03d}"
                seg_dir.mkdir(parents=True, exist_ok=True)
                for i, fr in enumerate(frames_selected[a:b]):
                    cv2.imwrite(str((seg_dir / f"frame_{i:04d}.jpg").as_posix()), fr)
                path_segment = str(seg_dir.as_posix())
            else:
                path_segment = ""  # vacío si no auditamos

            segment_uid = make_segment_uid(meta.video_id, win, seg_idx)

            meta_row = build_meta_row(
                meta,
                segment_uid=segment_uid, win=win, seg_idx=seg_idx, a=a, b=b,
                width=width, height=height, fps_target=fps_target,
                frames_ok_ratio=frames_ok_ratio, visible_hands_ratio=visible_hands_ratio,
                left_present_ratio=left_present_ratio, right_present_ratio=right_present_ratio,
                low_quality=low_quality,
                base_seg_dir=(base_seg_dir if base_seg_dir is not None else Path(path_segment)),
                stride=stride
            )
            # pisa path_segment si no estamos guardando frames
            if not save_frames:
                meta_row["path_segment"] = path_segment

            feat_row = build_feat_row(segment_uid, feats, feature_channels)

            meta_rows.append(meta_row)
            feat_rows.append(feat_row)

    return meta_rows, feat_rows


In [22]:
def save_outputs(all_meta_rows: List[dict], all_feat_rows: List[dict], out_root: Path):
    if not all_meta_rows:
        print("No se generaron segmentos.")
        return

    df_meta = pd.DataFrame(all_meta_rows)
    # map label_id SOLO en metadata
    classes = sorted(df_meta["label"].unique().tolist())
    class_to_id = {c: i for i, c in enumerate(classes)}
    df_meta["label_id"] = df_meta["label"].map(class_to_id)

    df_feat = pd.DataFrame(all_feat_rows)

    # orden consistente
    df_meta = df_meta[[c for c in META_COLS_FIXED if c in df_meta.columns]]
    df_feat = df_feat[select_feature_columns(df_feat, mode="all")]

    out_root.mkdir(parents=True, exist_ok=True)
    meta_path = out_root / "segments_metadata_train.csv"
    feat_path = out_root / "segments_features_train.csv"

    df_meta.to_csv(meta_path, index=False)
    df_feat.to_csv(feat_path, index=False)

    # opcional parquet
    try:
        df_meta.to_parquet(meta_path.with_suffix(".parquet"), index=False)
        df_feat.to_parquet(feat_path.with_suffix(".parquet"), index=False)
    except Exception:
        pass

    print(f"✔ METADATA: {meta_path} ({len(df_meta)} filas)")
    print(f"✔ FEATURES: {feat_path} ({len(df_feat)} filas, {df_feat.shape[1]-1} features)")
    print(f"Clases: {class_to_id}")


## Función de busqueda de videos

In [23]:
def find_videos(input_dirs: List[Path]) -> List[VideoMeta]:
    metas: List[VideoMeta] = []
    for root in input_dirs:
        if not root.exists():
            continue
        for p in sorted(root.rglob("*.mp4")):
            m = parse_video_name(p)
            if m is not None:
                metas.append(m)
    return metas

In [24]:
def load_train_matrices(meta_csv: str, feat_csv: str,
                        frames_ok_min=0.95, vis_min=0.90, drop_low_quality=True,
                        feature_mode="all"):
    df_meta = pd.read_csv(meta_csv)
    df_feat = pd.read_csv(feat_csv)

    # join mínimo: sólo lo que necesitas de meta
    keep_meta = ["segment_uid", "label_id", "person_id", "low_quality",
                 "frames_ok_ratio", "visible_hands_ratio"]
    df = df_feat.merge(df_meta[keep_meta], on="segment_uid", how="inner")

    # filtros de calidad
    if drop_low_quality:
        df = df[df["low_quality"] == 0]
    if frames_ok_min is not None:
        df = df[df["frames_ok_ratio"] >= frames_ok_min]
    if vis_min is not None:
        df = df[df["visible_hands_ratio"] >= vis_min]

    # features puras
    feat_cols = select_feature_columns(df, mode=feature_mode)
    feat_cols = [c for c in feat_cols if c != "segment_uid"]
    X = df[feat_cols].to_numpy(dtype=float)
    y = df["label_id"].to_numpy()
    groups = df["person_id"].to_numpy()
    return X, y, groups, df


## Main

In [25]:
def main():
    DERIV_ROOT.mkdir(parents=True, exist_ok=True)
    CSV_DIR.mkdir(parents=True, exist_ok=True)

    # crea FRAMES_OUT sólo si se va a usar
    frames_out_root = None
    if SAVE_WINDOWS_AS_FRAMES:
        frames_out_root = DERIV_ROOT / "frames"
        frames_out_root.mkdir(parents=True, exist_ok=True)

    videos = find_videos(INPUT_DIRS)
    print(f"Encontrados {len(videos)} videos válidos.")
    if not videos:
        print("No hay videos válidos. Revisa INPUT_DIRS.")
        return

    import mediapipe as mp
    Hands = mp.solutions.hands.Hands

    all_meta_rows: List[dict] = []
    all_feat_rows: List[dict] = []

    with Hands(static_image_mode=False, max_num_hands=2, model_complexity=1) as hands:
        for meta in tqdm(videos, desc="Procesando videos", unit="vid"):
            meta_rows, feat_rows = process_video(
                meta, hands,
                windows=WINDOWS,
                fps_target=FPS_TARGET,
                quality_thresh=QUALITY_THRESH,
                feature_channels=FEATURE_CHANNELS,
                save_frames=SAVE_WINDOWS_AS_FRAMES,
                frames_out_root=frames_out_root,
            )
            all_meta_rows.extend(meta_rows)
            all_feat_rows.extend(feat_rows)

    if not all_meta_rows:
        print("No se generaron segmentos.")
        return

    save_outputs(all_meta_rows, all_feat_rows, CSV_DIR)


In [26]:
if __name__ == "__main__":
    main()

Encontrados 517 videos válidos.


Procesando videos:   0%|          | 0/517 [00:00<?, ?vid/s]c:\Users\gael_\LSM_CICESE\Senas_Estaticas\venv_lsm\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
Procesando videos: 100%|██████████| 517/517 [29:57<00:00,  3.48s/vid]


✔ METADATA: derivatives\csv\segments_metadata_train.csv (46770 filas)
✔ FEATURES: derivatives\csv\segments_features_train.csv (46770 filas, 378 features)
Clases: {'J': 0, 'K': 1, 'Q': 2, 'X': 3, 'Z': 4, 'Ñ': 5}


# Entrenamiento Continuo de SVM

## Configuración

In [ ]:
CSV_IN = Path("derivatives/csv/segments_pooled_train.csv")
FILTER_LOW_QUALITY = True
STATS_ORDER = ["mean", "std", "min", "max", "p25", "p50", "p75", "dmean", "dstd"]
N_CHANNELS = 42  # 21 landmarks * (x,y)

C_GRID = [1, 3, 10, 30, 100]
GAMMA_GRID = ["scale", 0.3, 0.1, 0.03, 0.01]
N_SPLITS = 5  # GroupKFold por person_id

In [79]:
def make_feature_matrix(df: pd.DataFrame) -> np.ndarray:
    blocks = []
    for s in STATS_ORDER:
        cols = [f"{s}_f{i}" for i in range(N_CHANNELS)]
        blocks.append(df[cols].values)
    X = np.hstack(blocks).astype(np.float32)
    return X

In [80]:
def video_level_aggregate(df: pd.DataFrame, proba: np.ndarray, all_classes: np.ndarray) -> tuple[np.ndarray, np.ndarray, dict]:
    """
    Agrupa probabilidades por video y devuelve:
      - y_true_video (label_id reales)
      - y_pred_video (label_id reales, usando all_classes[argmax])
      - sizes: #segmentos por video usados en el promedio
    NOTA: 'proba' debe estar alineado fila-a-fila con df y sus columnas en el orden de 'all_classes'.
    """
    vids = df["video_id"].values
    y_true_seg = df["label_id"].values

    video_groups: dict[str, list] = {}
    video_targets: dict[str, int] = {}
    for i, vid in enumerate(vids):
        video_groups.setdefault(vid, []).append(proba[i])
        # Todos los segmentos de un video deberían tener el mismo label
        # Tomamos el primero que veamos (modo implícito).
        if vid not in video_targets:
            video_targets[vid] = int(y_true_seg[i])

    y_true_video, y_pred_video = [], []
    sizes = {}
    for vid, plist in video_groups.items():
        P = np.stack(plist, axis=0)           # (n_segs_video, n_classes_global)
        Pm = P.mean(axis=0)                   # promedio por clase
        pred_label = int(all_classes[np.argmax(Pm)])  # mapear índice->label_id real
        true_label = int(video_targets[vid])
        y_pred_video.append(pred_label)
        y_true_video.append(true_label)
        sizes[vid] = len(plist)

    return np.array(y_true_video), np.array(y_pred_video), sizes


In [81]:
def main():
    """
    Entrena y evalúa el SVM (Opción C) con validación por sujeto.
    Corrige el desajuste de dimensiones de 'predict_proba' mapeando a un orden global de clases.
    """
    import numpy as np
    import pandas as pd
    from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import Pipeline
    from sklearn.svm import SVC
    from sklearn.metrics import classification_report, confusion_matrix, f1_score

    if not CSV_IN.exists():
        print(f"No existe {CSV_IN}")
        return

    df = pd.read_csv(CSV_IN)

    # Diagnóstico de sujetos
    print("Sujetos únicos (antes):", df["person_id"].nunique())
    if FILTER_LOW_QUALITY and "low_quality" in df.columns:
        before = len(df)
        df = df[df["low_quality"] == 0].copy()
        print(f"Filtrado calidad: {before} -> {len(df)} segmentos")
    print("Sujetos únicos (después):", df["person_id"].nunique())
    if df["person_id"].nunique() < 2:
        raise ValueError("Se requieren ≥2 sujetos para CV por sujeto.")

    # Matriz de features y etiquetas
    X = make_feature_matrix(df)  # usa STATS_ORDER y N_CHANNELS definidos arriba
    y = df["label_id"].values
    groups = df["person_id"].values

    # Orden GLOBAL de clases (label_id reales)
    all_classes = np.unique(y)                 # p.ej., array([0,1,2,...,K-1]) o cualquier mapeo entero
    class_index = {c: i for i, c in enumerate(all_classes)}
    nC = all_classes.size

    # Selector de CV adaptativo
    def make_cv(groups_arr, desired_splits=5):
        n_groups = np.unique(groups_arr).size
        print(f"CV por sujeto: {n_groups} sujetos disponibles.")
        if n_groups >= desired_splits:
            print(f"Usando GroupKFold(n_splits={desired_splits})")
            return GroupKFold(n_splits=desired_splits)
        else:
            print("Usando LeaveOneGroupOut() (un sujeto fuera por fold)")
            return LeaveOneGroupOut()

    cv = make_cv(groups, desired_splits=N_SPLITS)

    # Pipeline SVM (probabilities=True para agregación a nivel video)
    base = Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("svm", SVC(kernel="rbf", class_weight="balanced", probability=True)),
    ])

    # Grid C×gamma con macro-F1 (segment-level OOF)
    best_cfg, best_f1 = None, -1.0
    for C in C_GRID:
        for gamma in GAMMA_GRID:
            fold_preds, fold_trues = [], []
            for tr_idx, te_idx in cv.split(X, y, groups):
                Xi, yi = X[tr_idx], y[tr_idx]
                Xj, yj = X[te_idx], y[te_idx]
                model = base.set_params(svm__C=C, svm__gamma=gamma)
                model.fit(Xi, yi)
                yhat = model.predict(Xj)
                fold_preds.append(yhat)
                fold_trues.append(yj)
            y_hat_all = np.concatenate(fold_preds)
            y_true_all = np.concatenate(fold_trues)
            f1 = f1_score(y_true_all, y_hat_all, average="macro")
            print(f"C={C:>5}, gamma={str(gamma):>6} -> f1_macro={f1:.4f}")
            if f1 > best_f1:
                best_f1, best_cfg = f1, (C, gamma)

    if best_cfg is None:
        raise RuntimeError("No se encontró configuración válida (C×gamma).")
    print("\nMejor configuración:", best_cfg, "f1_macro=", round(best_f1, 4))
    C_best, gamma_best = best_cfg

    # OOF final con proba REMAPEADA al orden global 'all_classes'
    y_true_all, y_pred_all, proba_all, idx_all = [], [], [], []
    for tr_idx, te_idx in cv.split(X, y, groups):
        Xi, yi = X[tr_idx], y[tr_idx]
        Xj, yj = X[te_idx], y[te_idx]
        model = base.set_params(svm__C=C_best, svm__gamma=gamma_best)
        model.fit(Xi, yi)

        yhat = model.predict(Xj)
        ypr  = model.predict_proba(Xj)  # columnas en el orden model.classes_

        # Remapear a matriz (n_test, nC) en el orden de 'all_classes'
        proba_fixed = np.zeros((len(te_idx), nC), dtype=float)
        cols = [class_index[c] for c in model.classes_]  # posiciones en all_classes
        proba_fixed[:, cols] = ypr

        y_true_all.append(yj)
        y_pred_all.append(yhat)
        proba_all.append(proba_fixed)
        idx_all.append(te_idx)

    y_true_all = np.concatenate(y_true_all)
    y_pred_all = np.concatenate(y_pred_all)
    proba_all  = np.concatenate(proba_all)   # ahora todas tienen nC columnas
    idx_all    = np.concatenate(idx_all)

    # Reordenar OOF para alinear con df
    order = np.argsort(idx_all)
    y_true_all = y_true_all[order]
    y_pred_all = y_pred_all[order]
    proba_all  = proba_all[order]

    # ===== Reporte global por segmento =====
    print("\n== Segment-level (OOF) ==")
    print(classification_report(y_true_all, y_pred_all, labels=all_classes, digits=4))
    print("Confusion matrix (segments):\n", confusion_matrix(y_true_all, y_pred_all, labels=all_classes))

    # ===== Reporte por tamaño de ventana =====
    for win in [30, 10, 5, 2]:
        mask = (df["window_size"].values == win)
        if not np.any(mask):
            continue
        print(f"\n== Segment-level por window_size={win} ==")
        print(classification_report(y_true_all[mask], y_pred_all[mask], labels=all_classes, digits=4))

    # ===== Reporte por vista =====
    for view in sorted(df["view"].astype(str).unique()):
        mask = (df["view"].astype(str).values == view)
        if not np.any(mask):
            continue
        print(f"\n== Segment-level view={view} ==")
        print(classification_report(y_true_all[mask], y_pred_all[mask], labels=all_classes, digits=4))

    # ===== Video-level (OOF) con probabilidades en orden global =====
    y_true_video, y_pred_video, sizes = video_level_aggregate(df, proba_all, all_classes)
    print("\n== Video-level (OOF, promedio de probabilidades) ==")
    print(classification_report(y_true_video, y_pred_video, labels=all_classes, digits=4))
    print("Confusion matrix (videos):\n", confusion_matrix(y_true_video, y_pred_video, labels=all_classes))

    # Diagnóstico por window_size a nivel video
    for win in [30, 10, 5, 2]:
        mask = (df["window_size"].values == win)
        if not np.any(mask):
            continue
        y_true_vw, y_pred_vw, _ = video_level_aggregate(df[mask].reset_index(drop=True), proba_all[mask], all_classes)
        print(f"\n== Video-level usando SOLO window_size={win} ==")
        print(classification_report(y_true_vw, y_pred_vw, labels=all_classes, digits=4))


In [ ]:
if __name__ == "__main__":
    main()

Sujetos únicos (antes): 18
Filtrado calidad: 46770 -> 46125 segmentos
Sujetos únicos (después): 18
CV por sujeto: 18 sujetos disponibles.
Usando GroupKFold(n_splits=5)
C=    1, gamma= scale -> f1_macro=0.8474
C=    1, gamma=   0.3 -> f1_macro=0.0634
